## Pattern 15: Union Find

### 🧠 What is Union-Find (Disjoint Set Union)?

**Union-Find** is a data structure that tracks a collection of **disjoint (non-overlapping) sets**. It supports two primary operations efficiently:

| Operation | What it does |
|-----------|-------------|
| **Find(x)** | Determine which set element `x` belongs to (returns the **root/representative**) |
| **Union(x, y)** | Merge the sets containing `x` and `y` into **one set** |

> **Think of it like friend groups** 👥 — Every person points to a "group leader." To check if two people are in the same group, you follow the chain to their leaders. To merge groups, you make one leader point to the other.

---

### 🔑 Core Idea

Each set is represented as a **tree**, where every node points to its **parent**. The **root** of the tree is the **representative** of that set.

```
Initial state (5 elements, each is its own set):
  0   1   2   3   4       ← each node is its own parent (root)
```

```
After union(0,1) and union(2,3):
  0       2
  |       |
  1       3       4       ← two groups formed, 4 still alone
```

```
After union(0,2):
      0
     / \
    1   2
        |
        3         4       ← three elements merged under root 0
```


---

### ⚡ Two Key Optimizations

Without optimizations, Union-Find can degrade to `O(n)` per operation. These two tricks make it nearly **O(1)**:

#### 1. Path Compression (in `find`)
When finding the root, **flatten the tree** by making every node on the path point directly to the root.
```
Before find(3):        After find(3):
    0                      0
    |                    / | \
    2                   1  2  3
    |
    3
```

#### 2. Union by Rank (in `union`)
Always attach the **shorter tree under the taller tree** to keep the overall tree shallow.
```
rank=1    rank=0           rank=1
  0         2      →        0
  |                        / \
  1                       1   2
```


Here's the **Markdown Cell 2 — Step-by-Step Walkthrough** again:

### 🪜 Step-by-Step Walkthrough

**Scenario:** 5 nodes (0–4). Perform: `union(0,1)`, `union(2,3)`, `union(1,3)`, then `find(3)`.

---

#### Initial State

```
parent: [0, 1, 2, 3, 4]   ← each node is its own parent
rank:   [0, 0, 0, 0, 0]   ← all ranks start at 0
Sets:   {0}, {1}, {2}, {3}, {4}
```

---

#### Step 1: `union(0, 1)`
- `find(0)` → root = **0**
- `find(1)` → root = **1**
- Roots differ → merge! Both rank 0 → attach 1 under 0, increment rank of 0.

```
parent: [0, 0, 2, 3, 4]
rank:   [1, 0, 0, 0, 0]
Sets:   {0,1}, {2}, {3}, {4}

Tree:   0
        |
        1
```

---

#### Step 2: `union(2, 3)`
- `find(2)` → root = **2**
- `find(3)` → root = **3**
- Roots differ → merge! Both rank 0 → attach 3 under 2, increment rank of 2.

```
parent: [0, 0, 2, 2, 4]
rank:   [1, 0, 1, 0, 0]
Sets:   {0,1}, {2,3}, {4}

Trees:  0       2
        |       |
        1       3
```

---

#### Step 3: `union(1, 3)`
- `find(1)` → follow 1→0, root = **0**
- `find(3)` → follow 3→2, root = **2**
- Roots differ → merge! Both rank 1 → attach 2 under 0, increment rank of 0.

```
parent: [0, 0, 0, 2, 4]
rank:   [2, 0, 1, 0, 0]
Sets:   {0,1,2,3}, {4}

Tree:     0
         / \
        1   2
            |
            3
```

---

#### Step 4: `find(3)` — Path Compression in Action!
- Follow path: 3 → 2 → 0 (root found!)
- **Path compression:** set parent of **3** directly to **0**

```
parent: [0, 0, 0, 0, 4]   ← node 3 now points directly to root 0

Tree:     0
        / | \
       1  2  3         ← flattened!
```

> After path compression, future `find(3)` calls are **O(1)** — it goes directly to the root.

### 🐍 Code Walkthrough

The class below implements Union-Find with **path compression** and **union by rank**:

- **`__init__(n)`** — Creates `n` elements, each in its own set. `parent[i] = i` and `rank[i] = 0`.
- **`find(x)`** — Recursively finds root of `x`. Path compression flattens tree via `self.parent[x] = self.find(self.parent[x])`.
- **`union(x, y)`** — Merges sets of `x` and `y`. Returns `False` if already in the same set. Uses rank to keep tree balanced.


In [1]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
    
    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    
    def union(self, x, y):
        px, py = self.find(x), self.find(y)
        if px == py:
            return False
        if self.rank[px] < self.rank[py]:
            px, py = py, px
        self.parent[py] = px
        if self.rank[px] == self.rank[py]:
            self.rank[px] += 1
        
        return True

### ▶️ Example Usage


In [2]:
# --- Example: Detecting connected components ---
uf = UnionFind(7)  # 7 nodes: 0 through 6

# Build connections
edges = [(0, 1), (1, 2), (3, 4), (5, 6), (4, 5)]

print("=== Performing Union Operations ===\n")
for u, v in edges:
    result = uf.union(u, v)
    status = "Merged" if result else "Already connected"
    print(f"union({u}, {v}) → {status}")
    print(f"  parent: {uf.parent}")
    print(f"  rank:   {uf.rank}\n")

# Check connectivity
print("=== Connectivity Queries ===\n")
queries = [(0, 2), (0, 3), (3, 6), (1, 5)]
for a, b in queries:
    connected = uf.find(a) == uf.find(b)
    symbol = "✅ Yes" if connected else "❌ No"
    print(f"Are {a} and {b} connected? {symbol}")

# Count distinct sets
num_components = len(set(uf.find(i) for i in range(7)))
print(f"\nNumber of connected components: {num_components}")

# --- Classic use case: Detect cycle in undirected graph ---
print("\n=== Cycle Detection Example ===\n")
uf2 = UnionFind(4)
graph_edges = [(0, 1), (1, 2), (2, 3), (3, 0)]  # last edge creates a cycle

for u, v in graph_edges:
    if uf2.find(u) == uf2.find(v):
        print(f"Edge ({u}, {v}) → 🔄 CYCLE DETECTED! Both already in same set.")
    else:
        uf2.union(u, v)
        print(f"Edge ({u}, {v}) → Added to graph.")


=== Performing Union Operations ===

union(0, 1) → Merged
  parent: [0, 0, 2, 3, 4, 5, 6]
  rank:   [1, 0, 0, 0, 0, 0, 0]

union(1, 2) → Merged
  parent: [0, 0, 0, 3, 4, 5, 6]
  rank:   [1, 0, 0, 0, 0, 0, 0]

union(3, 4) → Merged
  parent: [0, 0, 0, 3, 3, 5, 6]
  rank:   [1, 0, 0, 1, 0, 0, 0]

union(5, 6) → Merged
  parent: [0, 0, 0, 3, 3, 5, 5]
  rank:   [1, 0, 0, 1, 0, 1, 0]

union(4, 5) → Merged
  parent: [0, 0, 0, 3, 3, 3, 5]
  rank:   [1, 0, 0, 2, 0, 1, 0]

=== Connectivity Queries ===

Are 0 and 2 connected? ✅ Yes
Are 0 and 3 connected? ❌ No
Are 3 and 6 connected? ✅ Yes
Are 1 and 5 connected? ❌ No

Number of connected components: 2

=== Cycle Detection Example ===

Edge (0, 1) → Added to graph.
Edge (1, 2) → Added to graph.
Edge (2, 3) → Added to graph.
Edge (3, 0) → 🔄 CYCLE DETECTED! Both already in same set.


### ⏱️ Time & Space Complexity

#### Time Complexity: `O(α(n))` per operation ≈ **nearly O(1)**

| Operation | Without Optimizations | With Path Compression + Union by Rank |
|-----------|----------------------|---------------------------------------|
| `find(x)` | `O(n)` worst case | `O(α(n))` ≈ `O(1)` amortized |
| `union(x,y)` | `O(n)` worst case | `O(α(n))` ≈ `O(1)` amortized |
| `__init__(n)` | `O(n)` | `O(n)` |

> **`α(n)`** is the **Inverse Ackermann Function** — it grows *incredibly* slowly. For all practical purposes (`n < 10^80`), `α(n) ≤ 4`. So each operation is effectively **constant time**.

#### For `m` operations on `n` elements:
- **Total time:** `O(m · α(n))` ≈ `O(m)` in practice

#### Space Complexity: `O(n)`

| Component | Space |
|-----------|-------|
| `parent` array | `O(n)` |
| `rank` array | `O(n)` |
| Recursion stack (find) | `O(log n)` worst case before compression |
| **Total** | **`O(n)`** |

---

### 🌍 Classic Applications of Union-Find

| Problem | How Union-Find Helps |
|---------|---------------------|
| **Cycle Detection** (undirected graph) | If `find(u) == find(v)` before `union(u,v)`, there's a cycle |
| **Connected Components** | Each root represents a distinct component |
| **Kruskal's MST** | Use Union-Find to check if an edge connects two different components |
| **Network Connectivity** | Check if two computers/servers can communicate |
| **Accounts Merge** | Merge accounts sharing common emails |
| **Redundant Connections** | Find the extra edge that creates a cycle |

---

### 💡 Key Takeaway

> **Union-Find = "Group things together, and instantly check if two things are in the same group."**
> With path compression + union by rank, each operation is nearly **O(1)** — making it one of the most efficient data structures for connectivity problems.


### 📋 Notebook Structure After Adding
| # | Cell Type | Content |
|---|-----------|---------|
| 1	| Markdown | **Pattern 15:** Union Find |
| 2	| Markdown | Intuition & Concept |
| 3	| Markdown | Step-by-Step Walkthrough |
| 4	| Markdown | Code Walkthrough header |
| 5	| Code | `UnionFind` class |
| 6	| Markdown | Example Usage header |
| 7	| Code | Example execution + cycle detection |
| 8	| Markdown | Time/Space Complexity + Applications |